상용 API의 사용 여부 고려

In [1]:
# [역할] 실습 환경 준비: 프로젝트 루트로 이동하고, 재료·모델 경로를 정하고, backend를 import 경로에 추가
import os        # 작업 폴더 변경용
import pathlib   # 경로를 객체로 다루기
import sys       # 모듈 검색 경로(sys.path) 조작용

here = pathlib.Path.cwd()                                                  # 현재 작업 폴더
# ROOT = next(p for p in (here, *here.parents) if (p / 'backend').is_dir())  # 위로 올라가며 backend 폴더가 있는 곳 = 프로젝트 루트
ROOT = here.parents[2] if here.name == "Thu" else here      # Thu에서 실행했으면 3단계 위가 프로젝트 루트
os.chdir(ROOT)                                              # 작업 폴더를 루트로 이동
SANDBOX = ROOT / "sandbox" / "W5" / "Thu"                   # 오늘 실습 폴더
SAMPLES = ROOT / "sandbox" / "W5" / "Mon" / "samples"       # 샘플 문서(재료) 폴더
MODELS = ROOT.parent / 'models'                             # 임베딩 모델 폴더 (프로젝트 옆)
FORMATS = SAMPLES / '_formats'                              # 여러 형식 샘플 폴더

print("프로젝트 루트  :", ROOT)       # 루트 경로 확인
print("재료 폴더      :", SAMPLES)    # 샘플 폴더 경로 확인

if str(ROOT / 'backend') not in sys.path:          # backend가 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))      # 맨 앞에 추가 → `from app...` import 가능

프로젝트 루트  : d:\hanwha-agent
재료 폴더      : d:\hanwha-agent\sandbox\W5\Mon\samples


In [2]:
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local

scanned = FORMATS / "sample_scanned.pdf"
normal = SAMPLES / "DOC-HR-014_국내출장_여비_규정_v2.0.pdf" 

print(scanned.name)

try:
    parse_local(str(scanned))
except ValidationFailed as e:
    print(f" ValidationFailed : {e}")
    print(f" detail : {e.detail}")
print()

doc = parse_local(str(normal))
print(normal.name)
print(f"블록 {len(doc.blocks)}, 표 {doc.table_count}, 페이지 {doc.page_count}")


sample_scanned.pdf
15:48:36 INFO     app.rag.local_parsers: 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
 ValidationFailed : 문서에서 글자를 찾지 못했습니다: sample_scanned.pdf
 detail : 스캔본(이미지)일 수 있습니다. .env 의 UPSTAGE_PARSE_OCR 를 force 로 두고 실동작 모드로 다시 적재해 보세요

DOC-HR-014_국내출장_여비_규정_v2.0.pdf
블록 16, 표 0, 페이지 16


In [3]:
import pdfplumber
import pypdf


def pypdf_stat(path):
    reader = pypdf.PdfReader(str(path))
    return len(reader.pages), sum(len(pg.extract_text() or "") for pg in reader.pages)


def plumber_stat(path):
    with pdfplumber.open(str(path)) as pdf:
        chars = sum(len(pg.extract_text() or "") for pg in pdf.pages)
        tables = sum(len(pg.extract_tables()) for pg in pdf.pages)
        return len(pdf.pages), chars, tables 

print("scanned!")
a_pages, a_chars = pypdf_stat(scanned)
print(f" pypdf 페이지 {a_pages}, 글자 {a_chars}")
b_pages, b_chars, b_tables = plumber_stat(scanned)
print(f" plumber 페이지 {b_pages}, 글자 {b_chars}, 표 {b_tables}")

print()
print("normal!")
c_pages, _ = pypdf_stat(normal)
print(f" pypdf  페이지 {c_pages}, 표 {doc.table_count}")
d_pages, _, d_tables = plumber_stat(normal)
print(f" plumber  페이지 {d_pages}, 표 {d_tables}")


scanned!
 pypdf 페이지 1, 글자 0
 plumber 페이지 1, 글자 0, 표 0

normal!
 pypdf  페이지 16, 표 0
 plumber  페이지 16, 표 7
